In [17]:
# =============================================================================
# BLOCK 1: SETUP, IMPORTS, AND DATA LOADING
# =============================================================================
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torch.optim.lr_scheduler import OneCycleLR
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import KMeans
from sklearn.metrics import mean_squared_error
import joblib
import os
import gc

print("--- Step 1: Library Imports and Initial Setup ---")

# --- Global Constants & Paths ---
RANDOM_STATE = 42
N_SPLITS = 5
DATA_PATH = './'

# --- Load Raw Data ---
print("\n--- Step 2: Loading Raw Dataset and Test Files ---")
try:
    drop_cols = ['id', 'golf', 'view_rainier', 'view_skyline', 'view_lakesamm', 'view_otherwater', 'view_other']
    df_train = pd.read_csv(os.path.join(DATA_PATH, 'dataset.csv')).drop(columns=drop_cols)
    df_test = pd.read_csv(os.path.join(DATA_PATH, 'test.csv')).drop(columns=drop_cols)
    
    y_true = df_train['sale_price'].copy()
    grade_for_stratify = df_train['grade'].copy()
    
    print("Raw data loaded successfully. 'y_true' and 'grade_for_stratify' are ready.")
except FileNotFoundError as e:
    print(f"ERROR: Could not find data files. {e}")
    exit()

--- Step 1: Library Imports and Initial Setup ---

--- Step 2: Loading Raw Dataset and Test Files ---
Raw data loaded successfully. 'y_true' and 'grade_for_stratify' are ready.


In [18]:
# Make sure to have these libraries installed
# pip install pandas numpy scikit-learn

import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import KMeans
import gc

# Define a random state for reproducibility
RANDOM_STATE = 42

def create_comprehensive_features(df_train, df_test):
    """
    Combines original and new advanced feature engineering steps into a single pipeline.
    """
    print("--- Starting Comprehensive Feature Engineering ---")

    # Store original indices and target variable
    train_ids = df_train.index
    test_ids = df_test.index
    y_train = df_train['sale_price'].copy() # Keep the target separate

    # Combine for consistent processing
    df_train_temp = df_train.drop(columns=['sale_price'])
    all_data = pd.concat([df_train_temp, df_test], axis=0, ignore_index=True)

    # --- Original Feature Engineering ---

    # A) Brute-Force Numerical Interactions
    print("Step 1: Creating brute-force numerical interaction features...")
    NUMS = ['area', 'land_val', 'imp_val', 'sqft_lot', 'sqft', 'sqft_1', 'grade', 'year_built']
    # Ensure all columns exist and are numeric, fill missing with 0 for safety
    for col in NUMS:
        if col not in all_data.columns:
            all_data[col] = 0
        else:
            all_data[col] = pd.to_numeric(all_data[col], errors='coerce').fillna(0)
            
    for i in range(len(NUMS)):
        for j in range(i + 1, len(NUMS)):
            all_data[f'{NUMS[i]}_x_{NUMS[j]}'] = all_data[NUMS[i]] * all_data[NUMS[j]]

    # B) Date Features
    print("Step 2: Creating date features...")
    all_data['sale_date'] = pd.to_datetime(all_data['sale_date'])
    all_data['sale_year'] = all_data['sale_date'].dt.year
    all_data['sale_month'] = all_data['sale_date'].dt.month
    all_data['sale_dayofyear'] = all_data['sale_date'].dt.dayofyear
    all_data['age_at_sale'] = all_data['sale_year'] - all_data['year_built']

    # C) TF-IDF Text Features
    print("Step 3: Creating TF-IDF features for text columns...")
    text_cols = ['subdivision', 'zoning', 'city', 'sale_warning', 'join_status', 'submarket']
    all_data[text_cols] = all_data[text_cols].fillna('missing').astype(str)
    
    for col in text_cols:
        tfidf = TfidfVectorizer(analyzer='char', ngram_range=(3, 5), max_features=128, binary=True)
        svd = TruncatedSVD(n_components=8, random_state=RANDOM_STATE)
        
        tfidf_matrix = tfidf.fit_transform(all_data[col])
        tfidf_svd = svd.fit_transform(tfidf_matrix)
        
        tfidf_df = pd.DataFrame(tfidf_svd, columns=[f'{col}_tfidf_svd_{i}' for i in range(8)])
        all_data = pd.concat([all_data, tfidf_df], axis=1)

    # D) Log transform some interaction features
    for c in ['land_val_x_imp_val', 'land_val_x_sqft', 'imp_val_x_sqft']:
        if c in all_data.columns:
            all_data[c] = np.log1p(all_data[c].fillna(0))

    # --- New Feature Engineering Ideas ---

    # F) Group-By Aggregation Features
    print("Step 4: Creating group-by aggregation features...")
    group_cols = ['submarket', 'city', 'zoning']
    num_cols_for_agg = ['grade', 'sqft', 'imp_val', 'land_val', 'age_at_sale']

    for group_col in group_cols:
        for num_col in num_cols_for_agg:
            agg_stats = all_data.groupby(group_col)[num_col].agg(['mean', 'std', 'max', 'min']).reset_index()
            agg_stats.columns = [group_col] + [f'{group_col}_{num_col}_{stat}' for stat in ['mean', 'std', 'max', 'min']]
            all_data = pd.merge(all_data, agg_stats, on=group_col, how='left')
            all_data[f'{num_col}_minus_{group_col}_mean'] = all_data[num_col] - all_data[f'{group_col}_{num_col}_mean']

    # G) Ratio Features
    print("Step 5: Creating ratio features...")
    # Add a small epsilon to prevent division by zero
    epsilon = 1e-6 
    all_data['total_val'] = all_data['imp_val'] + all_data['land_val']
    all_data['imp_val_to_land_val_ratio'] = all_data['imp_val'] / (all_data['land_val'] + epsilon)
    all_data['land_val_ratio'] = all_data['land_val'] / (all_data['total_val'] + epsilon)
    all_data['sqft_to_lot_ratio'] = all_data['sqft'] / (all_data['sqft_lot'] + epsilon)
    all_data['was_renovated'] = (all_data['year_reno'] > 0).astype(int)
    all_data['reno_age_at_sale'] = np.where(all_data['was_renovated'] == 1, all_data['sale_year'] - all_data['year_reno'], -1)

    # H) Geospatial Clustering Features
    print("Step 6: Creating geospatial clustering features...")
    coords = all_data[['latitude', 'longitude']].copy()
    coords.fillna(coords.median(), inplace=True) # Simple imputation

    # KMeans is sensitive to feature scaling, but for lat/lon it's often okay without it.
    kmeans = KMeans(n_clusters=20, random_state=RANDOM_STATE, n_init=10) 
    all_data['location_cluster'] = kmeans.fit_predict(coords)
    
    # Calculate distance to each cluster center
    cluster_centers = kmeans.cluster_centers_
    for i in range(len(cluster_centers)):
        center = cluster_centers[i]
        all_data[f'dist_to_cluster_{i}'] = np.sqrt((coords['latitude'] - center[0])**2 + (coords['longitude'] - center[1])**2)


    # --- Final Cleanup ---
    print("Step 7: Finalizing feature set...")
    cols_to_drop = ['sale_date', 'subdivision', 'zoning', 'city', 'sale_warning', 'join_status', 'submarket']
    all_data = all_data.drop(columns=cols_to_drop)
    
    # One-hot encode the new cluster feature
    all_data = pd.get_dummies(all_data, columns=['location_cluster'], prefix='loc_cluster')
    
    # Final check for any remaining object columns
    object_cols = all_data.select_dtypes(include='object').columns
    if len(object_cols) > 0:
        all_data = all_data.drop(columns=object_cols)
        
    all_data.fillna(0, inplace=True)
    
    # === THE CRUCIAL FIX IS HERE ===
    # Convert ALL columns to a consistent numerical type to prevent errors.
    # float32 is memory-efficient and perfect for ML models.
    all_data = all_data.astype(np.float32)
    print("All feature columns successfully converted to float32.")


    # Separate back into train and test sets
    train_len = len(train_ids)
    X = all_data.iloc[:train_len].copy()
    X_test = all_data.iloc[train_len:].copy()
    
    # Restore original indices
    X.index = train_ids
    X_test.index = test_ids
    
    # Align columns - crucial for model prediction
    X_test = X_test[X.columns]
    
    print(f"\nComprehensive FE complete. Total features: {X.shape[1]}")
    gc.collect()
    
    return X, X_test, y_train
# =============================================================================
# BLOCK 2.5: EXECUTE FEATURE ENGINEERING
# =============================================================================
print("\n--- Starting Block 2.5: Executing Feature Engineering Pipeline ---")

# This is the crucial step that was missing.
# We call the function to create our training and testing dataframes.
X, X_test, y_train = create_comprehensive_features(df_train, df_test)

# Let's verify the output
print(f"Feature engineering complete. X shape: {X.shape}, X_test shape: {X_test.shape}")
gc.collect()

# =============================================================================
# BLOCK 2.6: ROBUST FEATURE CLIPPING (THE FINAL FIX)
# =============================================================================
print("\n--- Starting Block 2.6: Applying Robust Feature Clipping ---")
print("This step will protect the model from extreme outlier values in the test set.")

# We will iterate through each feature column
for col in X.columns:
    # Calculate the lower and upper bounds based ONLY on the training data distribution
    # Using the 0.1th and 99.9th percentiles is a robust way to handle outliers.
    lower_bound = X[col].quantile(0.001)
    upper_bound = X[col].quantile(0.999)
    
    # Count how many values in the test set are outside these bounds
    test_outliers = X_test[(X_test[col] < lower_bound) | (X_test[col] > upper_bound)][col].count()
    
    if test_outliers > 0:
        print(f"  - Feature '{col}': Found and clipped {test_outliers} extreme outliers.")
        
    # Apply the clipping to both the train and test sets
    # This ensures no value in the test set is outside the range seen in training.
    X[col] = X[col].clip(lower_bound, upper_bound)
    X_test[col] = X_test[col].clip(lower_bound, upper_bound)

print("\nRobust feature clipping complete. The data is now stabilized for scaling and training.")
gc.collect()


--- Starting Block 2.5: Executing Feature Engineering Pipeline ---
--- Starting Comprehensive Feature Engineering ---
Step 1: Creating brute-force numerical interaction features...
Step 2: Creating date features...
Step 3: Creating TF-IDF features for text columns...
Step 4: Creating group-by aggregation features...
Step 5: Creating ratio features...
Step 6: Creating geospatial clustering features...
Step 7: Finalizing feature set...
All feature columns successfully converted to float32.

Comprehensive FE complete. Total features: 233
Feature engineering complete. X shape: (200000, 233), X_test shape: (200000, 233)

--- Starting Block 2.6: Applying Robust Feature Clipping ---
This step will protect the model from extreme outlier values in the test set.
  - Feature 'sale_nbr': Found and clipped 113 extreme outliers.
  - Feature 'latitude': Found and clipped 384 extreme outliers.
  - Feature 'longitude': Found and clipped 400 extreme outliers.
  - Feature 'land_val': Found and clipped 1

0

In [19]:
# =============================================================================
# BLOCK 3: PYTORCH SETUP & FULL DATA PREPARATION
# =============================================================================
print(f"--- Starting Block 3: PyTorch Setup & Data Preparation ---")
print(f"PyTorch version: {torch.__version__}")
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# Data Scaling
print("\nScaling features and target variable for the Neural Network...")
feature_scaler = StandardScaler()
target_scaler = StandardScaler()
X_scaled = feature_scaler.fit_transform(X)
X_test_scaled = feature_scaler.transform(X_test)
y_true_scaled = target_scaler.fit_transform(y_true.to_numpy().reshape(-1, 1))

# Custom PyTorch Dataset
class HousePriceDataset(Dataset):
    def __init__(self, features, labels=None):
        self.features = features
        self.labels = labels
    def __len__(self): return len(self.features)
    def __getitem__(self, idx):
        features = torch.tensor(self.features[idx], dtype=torch.float32)
        if self.labels is not None:
            labels = torch.tensor(self.labels[idx], dtype=torch.float32)
            return features, labels
        return features
        
print("PyTorch setup and full data scaling complete.")

--- Starting Block 3: PyTorch Setup & Data Preparation ---
PyTorch version: 2.7.1+cu126
Using device: cuda

Scaling features and target variable for the Neural Network...
PyTorch setup and full data scaling complete.


In [20]:
# =============================================================================
# BLOCK 4: DEFINE THE STABILIZED RESIDUAL NEURAL NETWORK
# =============================================================================
class ResidualBlock(nn.Module):
    def __init__(self, input_size, output_size, dropout_rate):
        super(ResidualBlock, self).__init__()
        self.main_path = nn.Sequential(
            nn.Linear(input_size, output_size),
            nn.BatchNorm1d(output_size),
            nn.SiLU(),
            nn.Dropout(dropout_rate)
        )
        self.shortcut = nn.Identity() if input_size == output_size else nn.Linear(input_size, output_size)
    def forward(self, x): return self.main_path(x) + self.shortcut(x)

class ResidualNet(nn.Module):
    def __init__(self, input_shape, layer_sizes, dropout_rates):
        super(ResidualNet, self).__init__()
        layers = [nn.Linear(input_shape, layer_sizes[0]), nn.SiLU()]
        in_size = layer_sizes[0]
        for out_size, dropout in zip(layer_sizes, dropout_rates):
            layers.append(ResidualBlock(in_size, out_size, dropout))
            in_size = out_size
        layers.append(nn.Linear(in_size, 1))
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        output = self.model(x)
        # CRITICAL FIX #1: OUTPUT CLAMPING
        # Prevents the model from predicting impossibly large or small scaled values.
        return torch.clamp(output, -5, 5)

print("STABILIZED Residual Neural Network architecture defined successfully.")

STABILIZED Residual Neural Network architecture defined successfully.


In [21]:
# =============================================================================
# BLOCK 5 (FINAL & STABILIZED): DEFINE BEST HYPERPARAMETERS MANUALLY
# =============================================================================
print("--- Defining the best hyperparameters from the previous Optuna run ---")

# These are the exact results from your successful tuning process (Trial 23).
best_params_nn = {
    'learning_rate': 0.0007833213122687296, # The original rate from Optuna
    'weight_decay': 2.2353912564761073e-06,
    'layer_sizes': [512, 256, 128],
    'dropout_rates': [
        0.38831206715780076,
        0.1380700478405189,
        0.11587246202254503
    ]
}

# === THE FINAL FIX IS HERE ===
# We are manually overriding the learning rate with a more conservative value.
# This makes the training process more stable and prevents the weights from exploding.
# A common practice is to divide the Optuna-found rate by 3 to 5.
new_learning_rate = 0.0002 
best_params_nn['learning_rate'] = new_learning_rate

print("Successfully created the 'best_params_nn' dictionary.")
print(f"NOTE: The original learning rate was {0.000783:.6f}, but we are now using a more stable rate of {new_learning_rate}.")
print("Final parameters being used for training:")
print(best_params_nn)

--- Defining the best hyperparameters from the previous Optuna run ---
Successfully created the 'best_params_nn' dictionary.
NOTE: The original learning rate was 0.000783, but we are now using a more stable rate of 0.0002.
Final parameters being used for training:
{'learning_rate': 0.0002, 'weight_decay': 2.2353912564761073e-06, 'layer_sizes': [512, 256, 128], 'dropout_rates': [0.38831206715780076, 0.1380700478405189, 0.11587246202254503]}


In [22]:
# =============================================================================
# BLOCK 6: K-FOLD TRAINING WITH FORENSIC ANALYSIS, EVALUATION & SAVING
# =============================================================================
print("\n--- Starting Final K-Fold Training with Deep Analysis ---")

# --- Configuration & Initialization ---
EPOCHS, BATCH_SIZE, PATIENCE = 200, 512, 20
oof_nn_preds, test_nn_preds = np.zeros(len(X)), np.zeros(len(X_test))
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

# --- Main K-Fold Loop ---
for fold, (train_idx, val_idx) in enumerate(skf.split(X_scaled, grade_for_stratify)):
    print("\n" + "="*80)
    print(f"--- FORENSIC ANALYSIS: TRAINING FOLD {fold+1}/{N_SPLITS} ---")
    print("="*80)

    # Setup DataLoaders, Model, and Optimizer
    X_train, y_train = X_scaled[train_idx], y_true_scaled[train_idx]
    X_val, y_val = X_scaled[val_idx], y_true_scaled[val_idx]
    train_loader = DataLoader(HousePriceDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(HousePriceDataset(X_val, y_val), batch_size=BATCH_SIZE, shuffle=False)
    model = ResidualNet(
        input_shape=X_train.shape[1],
        layer_sizes=best_params_nn['layer_sizes'],
        dropout_rates=best_params_nn['dropout_rates']
    ).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=best_params_nn['learning_rate'], weight_decay=best_params_nn['weight_decay'])
    scheduler = OneCycleLR(optimizer, max_lr=best_params_nn['learning_rate'], epochs=EPOCHS, steps_per_epoch=len(train_loader))
    loss_fn = nn.HuberLoss()
    best_val_loss, best_model_state = float('inf'), None

    # Training & Validation Loop
    for epoch in range(EPOCHS):
        model.train()
        for features, labels in train_loader:
            features, labels = features.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = loss_fn(model(features), labels)
            loss.backward()
            # CRITICAL FIX #2: GRADIENT CLIPPING
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
        
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for features, labels in val_loader:
                val_loss += loss_fn(model(features.to(device)), labels.to(device)).item()
        val_loss /= len(val_loader)
        
        if (epoch + 1) % 20 == 0: print(f"  Epoch {epoch+1:03d} | Val Loss: {val_loss:.6f}")
        
        if val_loss < best_val_loss:
            best_val_loss, epochs_no_improve, best_model_state = val_loss, 0, model.state_dict().copy()
        else:
            epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f"\nEarly stopping at epoch {epoch+1}. Best validation loss: {best_val_loss:.6f}")
            break

    # --- Generate and Inspect Predictions ---
    print(f"\nFold {fold+1} training complete. Generating and inspecting predictions...")
    model.load_state_dict(best_model_state)
    model.eval()

    # OOF Predictions
    with torch.no_grad():
        raw_oof_preds = np.concatenate([model(f.to(device)).detach().cpu().numpy() for f, _ in val_loader])
    oof_nn_preds[val_idx] = target_scaler.inverse_transform(raw_oof_preds).flatten()

    # Test Predictions with Forensic Analysis
    test_loader_fold = DataLoader(HousePriceDataset(X_test_scaled), batch_size=BATCH_SIZE*2, shuffle=False)
    with torch.no_grad():
        raw_test_preds = np.concatenate([model(f.to(device)).detach().cpu().numpy() for f in test_loader_fold])
    
    print("\n" + "-"*25 + f" ANALYSIS FOR FOLD {fold+1} " + "-"*25)
    print("Describing RAW (SCALED) test predictions (direct model output):")
    print(pd.Series(raw_test_preds.flatten()).describe())
    
    inversed_test_preds = target_scaler.inverse_transform(raw_test_preds).flatten()
    print("\nDescribing INVERSE-TRANSFORMED test predictions for this fold:")
    print(pd.Series(inversed_test_preds).describe())
    
    if np.min(inversed_test_preds) < -10000:
        print(f"\n*** BUG DETECTED IN FOLD {fold+1}: Extreme negative values found. ***")
    else:
        print(f"\n  Fold {fold+1} predictions appear clean.")
    print("-"*65)
        
    test_nn_preds += inversed_test_preds
    del model, X_train, X_val, y_train, y_val, train_loader, val_loader, test_loader_fold, raw_test_preds
    gc.collect()

# --- 4. Finalize, Evaluate, and Save ---
print("\n" + "="*80)
print("--- K-Fold Training Complete: Final Analysis and Saving ---")
print("="*80)

test_nn_preds /= N_SPLITS
final_mean_rmse_nn = np.sqrt(mean_squared_error(y_true, oof_nn_preds))
print(f"Final NN Mean Model OOF RMSE: ${final_mean_rmse_nn:,.2f}")

print("\n--- Final Forensic Analysis of FINAL AVERAGED Prediction Arrays ---")
print("Describing final 'oof_nn_preds':\n", pd.Series(oof_nn_preds).describe())
print("\nDescribing final 'test_nn_preds':\n", pd.Series(test_nn_preds).describe())

if np.isnan(test_nn_preds).any() or np.min(test_nn_preds) < -10000:
    print("\n*** CRITICAL WARNING: Final averaged test predictions are still corrupt! ***")
else:
    print("\nSUCCESS: Final averaged test predictions appear clean and reasonable.")




--- Starting Final K-Fold Training with Deep Analysis ---

--- FORENSIC ANALYSIS: TRAINING FOLD 1/5 ---
  Epoch 020 | Val Loss: 0.043797
  Epoch 040 | Val Loss: 0.036632
  Epoch 060 | Val Loss: 0.037957
  Epoch 080 | Val Loss: 0.032446
  Epoch 100 | Val Loss: 0.031988
  Epoch 120 | Val Loss: 0.032634
  Epoch 140 | Val Loss: 0.031797
  Epoch 160 | Val Loss: 0.031592

Early stopping at epoch 170. Best validation loss: 0.031277

Fold 1 training complete. Generating and inspecting predictions...

------------------------- ANALYSIS FOR FOLD 1 -------------------------
Describing RAW (SCALED) test predictions (direct model output):
count    200000.000000
mean          0.001996
std           0.997918
min          -1.355245
25%          -0.670281
50%          -0.294274
75%           0.326076
max           5.000000
dtype: float64

Describing INVERSE-TRANSFORMED test predictions for this fold:
count    2.000000e+05
mean     5.849821e+05
std      4.161899e+05
min      1.893299e+04
25%      3.046

In [23]:
print("\n--- Step 5: Saving Prediction Arrays ---")
SAVE_PATH = './NN_model_predictions/'
os.makedirs(SAVE_PATH, exist_ok=True)
np.save(os.path.join(SAVE_PATH, 'oof_nn_preds.npy'), oof_nn_preds)
np.save(os.path.join(SAVE_PATH, 'test_nn_preds.npy'), test_nn_preds)
print(f"Prediction arrays saved successfully to: {SAVE_PATH}")


--- Step 5: Saving Prediction Arrays ---
Prediction arrays saved successfully to: ./NN_model_predictions/
